### setup dependencies

In [14]:
%use ktor-client, serialization, dataframe

In [15]:
import io.ktor.http.*
import io.ktor.client.request.*
import io.ktor.client.request.forms.*
import io.ktor.client.statement.*
import kotlinx.serialization.*
import kotlinx.serialization.json.*
import kotlinx.coroutines.runBlocking

import org.jetbrains.kotlinx.dataframe.api.*
import org.jetbrains.kotlinx.dataframe.*

val client = http.ktorClient

val json = Json {
    ignoreUnknownKeys = true
    isLenient = true
    prettyPrint = true
}

### config

In [16]:
val realmBase = "http://localhost:7766/realms/flagforge"
val wellKnownUrl = "$realmBase/.well-known/openid-configuration"
val tokenUrl = "$realmBase/protocol/openid-connect/token"

val userInfoUrl = "http://localhost:8085/userinfo"

val clientId = "flagforge-frontend"
val username = "ff-admin"
val password = "ff"

println(
    """
    Test config:
      realmBase   = $realmBase
      tokenUrl    = $tokenUrl
      userInfoUrl = $userInfoUrl
      clientId    = $clientId
      username    = $username
    """.trimIndent()
)

Test config:
  realmBase   = http://localhost:7766/realms/flagforge
  tokenUrl    = http://localhost:7766/realms/flagforge/protocol/openid-connect/token
  userInfoUrl = http://localhost:8085/userinfo
  clientId    = flagforge-frontend
  username    = ff-admin


### Discovery: .well-known

In [17]:
val discoveryJson = runBlocking {
    val response = client.get(wellKnownUrl)
    check(response.status.isSuccess()) {
        "Realm discovery failed: ${response.status}"
    }
    println("Keycloak realm is reachable: ${response.status}")
    Json.parseToJsonElement(response.bodyAsText()).jsonObject
}

data class OidcEndpoint(val name: String, val url: String?)

val endpoints = listOf(
    OidcEndpoint("issuer", discoveryJson["issuer"]?.jsonPrimitive?.content),
    OidcEndpoint("authorization_endpoint", discoveryJson["authorization_endpoint"]?.jsonPrimitive?.content),
    OidcEndpoint("token_endpoint", discoveryJson["token_endpoint"]?.jsonPrimitive?.content),
    OidcEndpoint("userinfo_endpoint", discoveryJson["userinfo_endpoint"]?.jsonPrimitive?.content),
    OidcEndpoint("jwks_uri", discoveryJson["jwks_uri"]?.jsonPrimitive?.content)
)

val discoveryDf = endpoints.toDataFrame()
discoveryDf

Keycloak realm is reachable: 200 OK


name,url
issuer,http://localhost:7766/realms/flagforge
authorization_endpoint,http://localhost:7766/realms/flagforg...
token_endpoint,http://localhost:7766/realms/flagforg...
userinfo_endpoint,http://localhost:7766/realms/flagforg...
jwks_uri,http://localhost:7766/realms/flagforg...


### Got access token

In [18]:
@Serializable
data class TokenResponse(
    val access_token: String,
    val expires_in: Long? = null,
    val refresh_token: String? = null,
    val refresh_expires_in: Long? = null,
    val scope: String? = null,
    val token_type: String? = null,
    @SerialName("not-before-policy")
    val not_before_policy: Int? = null,
    val session_state: String? = null,
)

suspend fun <T> measureMs(label: String, block: suspend () -> T): T {
    val start = System.currentTimeMillis()
    val result = block()
    val duration = System.currentTimeMillis() - start
    println("⏱ $label took ${duration} ms")
    return result
}

suspend fun fetchToken(): TokenResponse =
    measureMs("Token request") {
        val response = client.submitForm(
            url = tokenUrl,
            formParameters = Parameters.build {
                append("grant_type", "password")
                append("client_id", clientId)
                append("username", username)
                append("password", password)
            }
        )

        check(response.status.isSuccess()) {
            "Token endpoint failed: ${response.status}, body=${response.bodyAsText()}"
        }

        val tokenJson = Json.parseToJsonElement(response.bodyAsText())
        val token = Json.decodeFromJsonElement<TokenResponse>(tokenJson)

        require(token.access_token.isNotBlank()) {
            "access_token is blank in token response: $tokenJson"
        }

        println("Got access token (${token.access_token.take(16)}...)")
        token
    }

val tokenResponse = runBlocking { fetchToken() }
val accessToken = tokenResponse.access_token

data class TokenSummary(
    val token_preview: String,
    val expires_in_sec: Long?,
    val has_refresh_token: Boolean,
    val scope: String?,
    val token_type: String?
)

val tokenDf = listOf(
    TokenSummary(
        token_preview = accessToken.take(32) + "…",
        expires_in_sec = tokenResponse.expires_in,
        has_refresh_token = !tokenResponse.refresh_token.isNullOrBlank(),
        scope = tokenResponse.scope,
        token_type = tokenResponse.token_type
    )
).toDataFrame()

tokenDf

Got access token (eyJhbGciOiJSUzI1...)
⏱ Token request took 731 ms


token_preview,expires_in_sec,has_refresh_token,scope,token_type
eyJhbGciOiJSUzI1NiIsInR5cCIgOiAi…,300,true,email profile,Bearer


### call /userinfo in server-token

In [19]:
val userInfoJson = runBlocking {
    val response = client.get(userInfoUrl) {
        header(HttpHeaders.Authorization, "Bearer $accessToken")
    }

    check(response.status.isSuccess()) {
        "userinfo failed: ${response.status}, body=${response.bodyAsText()}"
    }

    println("/userinfo responded with 200 OK")
    Json.parseToJsonElement(response.bodyAsText()).jsonObject
}

@OptIn(ExperimentalSerializationApi::class)
fun JsonElement.pretty(): String =
    Json {
        prettyPrint = true
        prettyPrintIndent = "  "
    }.encodeToString(this)

val pretty = json.encodeToString(userInfoJson)
println(userInfoJson.pretty())

/userinfo responded with 200 OK
{
  "sub": "b83220c6-a489-4f2f-a088-8949164cc7ec",
  "resource_access": {
    "flagforge-cloud-gateway": {
      "roles": [
        "api_admin"
      ]
    }
  },
  "email_verified": true,
  "allowed-origins": [
    "http://localhost:3000"
  ],
  "iss": "http://localhost:7766/realms/flagforge",
  "typ": "Bearer",
  "preferred_username": "ff-admin",
  "given_name": "Flagforge",
  "sid": "3d92290d-43cc-12df-c9f6-41abaf78208e",
  "aud": [
    "flagforge-cloud-gateway"
  ],
  "acr": "1",
  "realm_access": {
    "roles": [
      "ff_user",
      "ff_admin"
    ]
  },
  "azp": "flagforge-frontend",
  "scope": "email profile",
  "name": "Flagforge Admin",
  "exp": "2025-11-25T19:24:50Z",
  "iat": "2025-11-25T19:19:50Z",
  "family_name": "Admin",
  "jti": "onrtro:3b08bd9d-78cc-aa37-6470-09e03661e1a3",
  "email": "ff-admin@example.local",
  "roles": [
    "flagforge-cloud-gateway_api_admin",
    "ff_user",
    "ff_admin"
  ]
}


### claims visualize

In [20]:
data class ClaimRow(
    val claim: String,
    val kind: String,
    val value: String,
)

val claimRows = userInfoJson.entries.map { (key, value) ->
    val kind = when (value) {
        is JsonPrimitive -> when {
            value.isString -> "string"
            value.booleanOrNull != null -> "boolean"
            value.longOrNull != null -> "number"
            else -> "primitive"
        }
        is JsonArray -> "array"
        is JsonObject -> "object"
        else -> "unknown"
    }

    ClaimRow(
        claim = key,
        kind = kind,
        value = value.toString()
    )
}

val claimsDf = claimRows.toDataFrame()
claimsDf

claim,kind,value
sub,string,"""b83220c6-a489-4f2f-a088-8949164cc7ec"""
resource_access,object,"{""flagforge-cloud-gateway"":{""roles"":[..."
email_verified,boolean,true
allowed-origins,array,"[""http://localhost:3000""]"
iss,string,"""http://localhost:7766/realms/flagforge"""
typ,string,"""Bearer"""
preferred_username,string,"""ff-admin"""
given_name,string,"""Flagforge"""
sid,string,"""3d92290d-43cc-12df-c9f6-41abaf78208e"""
aud,array,"[""flagforge-cloud-gateway""]"


### roles visualize

In [21]:
val preferredUsername = userInfoJson["preferred_username"]?.jsonPrimitive?.content

val rolesJson = userInfoJson["roles"] ?: JsonArray(emptyList())
val roles = rolesJson.jsonArray.mapNotNull { it.jsonPrimitive.contentOrNull }

data class RoleRow(val role: String)

val rolesDf = roles.map(::RoleRow).toDataFrame()

println("preferred_username = $preferredUsername")
println("roles (${roles.size}): $roles")

rolesDf

preferred_username = ff-admin
roles (3): [flagforge-cloud-gateway_api_admin, ff_user, ff_admin]


role
flagforge-cloud-gateway_api_admin
ff_user
ff_admin


### check-up

In [22]:
check(preferredUsername == username) {
    "preferred_username in /userinfo doesn't match test username '$username', got '$preferredUsername'"
}

require(userInfoJson["roles"] != null) {
    "roles claim was not added by server-token"
}

require(roles.isNotEmpty()) {
    "roles array is empty in /userinfo response"
}

val issuerFromDiscovery = discoveryJson["issuer"]?.jsonPrimitive?.content
val issFromUserInfo = userInfoJson["iss"]?.jsonPrimitive?.content

if (issuerFromDiscovery != null && issFromUserInfo != null) {
    check(issuerFromDiscovery == issFromUserInfo) {
        "Issuer mismatch: discovery.issuer='$issuerFromDiscovery', userinfo.iss='$issFromUserInfo'"
    }
}

println("🎉 End-to-end test passed: Keycloak + server-token + example-realm works as expected")
println("   - Discovery OK")
println("   - Token endpoint OK")
println("   - /userinfo OK")
println("   - Claims & roles validated")

🎉 End-to-end test passed: Keycloak + server-token + example-realm works as expected
   - Discovery OK
   - Token endpoint OK
   - /userinfo OK
   - Claims & roles validated
